In [1]:
# Import libraries

import pandas as pd
import numpy as np

In [2]:
# Load cleaned dataset

df = pd.read_csv(
    "../data/processed/cleaned_transactions.csv"
)

# Display first 5 rows
df.head()

,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,new_device,...,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud,hour,day_of_week,month,year,amount_dest
0,2022-10-03 18:40:59.468549+00:00,us,usd,cad,atm,278.19,278.19,4.25,1.351351,0,...,0.223,0,0,0.0,0,18,Monday,10,2022,375.93
1,2022-10-03 20:39:38.468549+00:00,ca,cad,mxn,web,208.51,154.29,4.24,12.758621,1,...,0.268,0,1,0.0,0,20,Monday,10,2022,2660.30
2,2022-10-03 23:02:43.468549+00:00,us,usd,cny,mobile,160.33,160.33,2.70,7.142857,0,...,0.176,0,0,0.0,0,23,Monday,10,2022,1145.21
3,2022-10-04 01:08:53.468549+00:00,us,usd,eur,mobile,59.41,59.41,2.22,0.925926,0,...,0.391,0,0,0.0,0,1,Tuesday,10,2022,55.01
4,2022-10-04 09:35:03.468549+00:00,us,usd,inr,mobile,200.96,200.96,3.61,83.333333,0,...,0.257,0,0,0.0,0,9,Tuesday,10,2022,16746.67


In [3]:
# Convert timestamp column to datetime
# This is needed for chronological train-test splitting

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

In [4]:
# Sort dataset chronologically by transaction timestamp
# This ensures older transactions are used for training
# and newer transactions are used for testing

df = df.sort_values(
    by="timestamp"
).reset_index(drop=True)

In [5]:
# Check transaction date range

print("Start date:", df["timestamp"].min())
print("End date:", df["timestamp"].max())

Start date: 2022-10-03 18:40:59.468549+00:00
End date: 2025-12-16 00:13:41.468549+00:00


In [6]:
# Identify categorical columns for encoding
# Keep timestamp for now because it is still needed for chronological splitting

categorical_cols = [
    "home_country",
    "source_currency",
    "dest_currency",
    "channel",
    "ip_country",
    "kyc_tier",
    "day_of_week"
]

In [7]:
# One-hot encode categorical variables
# drop_first=True removes one category from each feature
# to reduce redundant columns and avoid multicollinearity

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

In [8]:
# Convert boolean dummy variables to integers
# This ensures encoded features are numeric

bool_cols = df.select_dtypes(
    include=["bool"]
).columns

df[bool_cols] = df[bool_cols].astype(int)

In [9]:
# Define chronological split index
# First 80% of records will be used for training
# Last 20% of records will be used for testing

split_index = int(
    len(df) * 0.8
)

print("Split index:", split_index)

Split index: 8752


In [10]:
# Split the dataset chronologically
# Timestamp is still present here to preserve time order

train_df = df.iloc[:split_index].copy()

test_df = df.iloc[split_index:].copy()

In [11]:
# Check date range for training and testing sets

print("Training start date:", train_df["timestamp"].min())
print("Training end date:", train_df["timestamp"].max())

print("\nTesting start date:", test_df["timestamp"].min())
print("Testing end date:", test_df["timestamp"].max())

Training start date: 2022-10-03 18:40:59.468549+00:00
Training end date: 2025-03-23 15:23:26.468549+00:00

Testing start date: 2025-03-23 20:11:35.468549+00:00
Testing end date: 2025-12-16 00:13:41.468549+00:00


In [12]:
# Separating features and target variable
# Droping timestamp, after chronological splitting
# The model uses engineered time features such as hour, day_of_week, month and year

X_train = train_df.drop(
    columns=["is_fraud", "timestamp"]
)

y_train = train_df["is_fraud"]

X_test = test_df.drop(
    columns=["is_fraud", "timestamp"]
)

y_test = test_df["is_fraud"]

In [13]:
# Checking feature and target shapes

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (8752, 46)
X_test shape: (2188, 46)
y_train shape: (8752,)
y_test shape: (2188,)


In [14]:
# Confirming all feature columns are numeric and model-ready

X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 8752 entries, 0 to 8751
Data columns (total 46 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   amount_src                 8752 non-null   float64
 1   amount_usd                 8752 non-null   float64
 2   fee                        8752 non-null   float64
 3   exchange_rate_src_to_dest  8752 non-null   float64
 4   new_device                 8752 non-null   int64  
 5   location_mismatch          8752 non-null   int64  
 6   ip_risk_score              8752 non-null   float64
 7   account_age_days           8752 non-null   int64  
 8   device_trust_score         8752 non-null   float64
 9   chargeback_history_count   8752 non-null   int64  
 10  risk_score_internal        8752 non-null   float64
 11  txn_velocity_1h            8752 non-null   int64  
 12  txn_velocity_24h           8752 non-null   int64  
 13  corridor_risk              8752 non-null   float64
 14  hou

In [15]:
# Check fraud distribution in chronological training and testing sets

print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target percentage:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nTesting target percentage:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training target distribution:
is_fraud
0    8072
1     680
Name: count, dtype: int64

Training target percentage:
is_fraud
0    92.23
1     7.77
Name: proportion, dtype: float64

Testing target distribution:
is_fraud
0    1879
1     309
Name: count, dtype: int64

Testing target percentage:
is_fraud
0    85.88
1    14.12
Name: proportion, dtype: float64


In [16]:
# Save processed training and testing datasets

X_train.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

In [17]:
# Confirming files were saved successfully

print("Feature engineering completed successfully.")
print("Chronological training and testing datasets saved to data/processed/")

Feature engineering completed successfully.
Chronological training and testing datasets saved to data/processed/


## Feature Engineering Summary

- The cleaned dataset was loaded from the processed data folder.
- The timestamp column was converted to datetime format and used to sort transactions chronologically.
- A chronological train-test split was applied, using the earliest 80% of transactions for training and the latest 20% for testing.
- This approach better reflects real-world fraud detection, where models are trained on past transactions and used to predict future transactions.
- The raw timestamp column was removed only after splitting to avoid losing temporal ordering.
- Time-based features such as hour, day_of_week, month and year were retained for modelling.
- Categorical variables were converted into numerical format using one-hot encoding.
- Boolean dummy variables were converted into integer format.
- Processed training and testing datasets were saved for model training.